In [1]:
import datasets
import transformers

import shap

/home/user/Source/kuznetsovmd/privacy-ontology/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = datasets.load_dataset("imdb", split="test")

# shorten the strings to fit into the pipeline model
short_data = [v[:500] for v in dataset["text"][:20]]
dataset[0]

Generating unsupervised split: 100%|██████████| 50000/50000 [00:04<00:00, 10607.18 examples/s]


{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as 

In [3]:
classifier = transformers.pipeline("sentiment-analysis", return_all_scores=True)
classifier(short_data[:2])

No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
config.json: 100%|██████████| 629/629 [00:00<00:00, 3.18MB/s]
model.safetensors: 100%|██████████| 268M/268M [00:24<00:00, 11.0MB/s] 
tokenizer_config.json: 100%|██████████| 48.0/48.0 [00:00<00:00, 530kB/s]
vocab.txt: 100%|██████████| 232k/232k [00:00<00:00, 1.72MB/s]
`return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.


[[{'label': 'NEGATIVE', 'score': 0.07582096010446548},
  {'label': 'POSITIVE', 'score': 0.9241790771484375}],
 [{'label': 'NEGATIVE', 'score': 0.018342578783631325},
  {'label': 'POSITIVE', 'score': 0.9816573858261108}]]

In [4]:
# define the explainer
explainer = shap.Explainer(classifier)

In [5]:
# explain the predictions of the pipeline on the first two samples
shap_values = explainer(short_data[:3])

PartitionExplainer explainer: 4it [01:02, 20.87s/it]                       


In [6]:
data = shap_values.data[0]
data

array(['', 'I ', 'love ', 'sci', '-', 'fi ', 'and ', 'am ', 'willing ',
       'to ', 'put ', 'up ', 'with ', 'a ', 'lot', '. ', 'Sci', '-',
       'fi ', 'movies', '/', 'TV ', 'are ', 'usually ', 'under', 'fu',
       'nded', ', ', 'under', '-', 'appreciated ', 'and ',
       'misunderstood', '. ', 'I ', 'tried ', 'to ', 'like ', 'this',
       ', ', 'I ', 'really ', 'did', ', ', 'but ', 'it ', 'is ', 'to ',
       'good ', 'TV ', 'sci', '-', 'fi ', 'as ', 'Babylon ', '5 ', 'is ',
       'to ', 'Star ', 'Trek ', '(', 'the ', 'original', ')', '. ',
       'Silly ', 'pro', 'st', 'hetic', 's', ', ', 'cheap ', 'cardboard ',
       'sets', ', ', 'stil', 'ted ', 'dialogues', ', ', 'C', 'G ',
       'that ', 'doesn', "'", 't ', 'match ', 'the ', 'background', ', ',
       'and ', 'painfully ', 'one', '-', 'dimensional ', 'characters ',
       'cannot ', 'be ', 'overcome ', 'with ', 'a ', "'", 'sci', '-',
       'fi', "' ", 'setting', '. ', '(', 'I', "'", 'm ', 'sure ',
       'there ', 'are 

In [7]:
[(data[i], v) for i, v in enumerate(shap_values.values[0][:,1])]

[('', 0.010352682624905955),
 ('I ', 0.010352682624905955),
 ('love ', 0.022449701395779502),
 ('sci', 0.022449701395779502),
 ('-', 0.0040818209255048645),
 ('fi ', 0.0040818209255048645),
 ('and ', 0.0040818209255048645),
 ('am ', 0.001853318198080759),
 ('willing ', 0.001853318198080759),
 ('to ', 0.001853318198080759),
 ('put ', 0.001853318198080759),
 ('up ', 0.0018533181980807591),
 ('with ', 0.0018533181980807591),
 ('a ', 0.0018533181980807591),
 ('lot', 0.0018533181980807591),
 ('. ', 0.0018533181980807591),
 ('Sci', -0.0017312866238244243),
 ('-', -0.0017312866238244243),
 ('fi ', -0.0017312866238244243),
 ('movies', -0.0017312866238244243),
 ('/', -0.0017312866238244243),
 ('TV ', -0.0017312866238244243),
 ('are ', -0.0017312866238244243),
 ('usually ', -0.0017312866238244243),
 ('under', -0.003898118467903267),
 ('fu', -0.003898118467903267),
 ('nded', -0.003898118467903267),
 (', ', -0.003898118467903267),
 ('under', -0.01673762774399439),
 ('-', -0.016170997585421),
 ('ap

In [8]:
shap_values

.values =
array([array([[-0.01035268,  0.01035268],
              [-0.01035268,  0.01035268],
              [-0.0224497 ,  0.0224497 ],
              [-0.0224497 ,  0.0224497 ],
              [-0.00408182,  0.00408182],
              [-0.00408182,  0.00408182],
              [-0.00408182,  0.00408182],
              [-0.00185332,  0.00185332],
              [-0.00185332,  0.00185332],
              [-0.00185332,  0.00185332],
              [-0.00185332,  0.00185332],
              [-0.00185332,  0.00185332],
              [-0.00185332,  0.00185332],
              [-0.00185332,  0.00185332],
              [-0.00185332,  0.00185332],
              [-0.00185332,  0.00185332],
              [ 0.00173129, -0.00173129],
              [ 0.00173129, -0.00173129],
              [ 0.00173129, -0.00173129],
              [ 0.00173129, -0.00173129],
              [ 0.00173129, -0.00173129],
              [ 0.00173129, -0.00173129],
              [ 0.00173129, -0.00173129],
              [ 0.001731

In [9]:
[sum(shap_values.values[i][:,1]) + shap_values.base_values[i] for i in range(3)]

[array([1.0499613 , 0.92417908]),
 array([1.05931389, 0.98165739]),
 array([0.0947141 , 0.00052696])]

In [10]:
shap.text_plot(shap_values)